# Simple Pipeline: A 3-Stage Agent Pipeline

This example demonstrates the fundamental pattern of agent pipelines:
the output of one agent becomes the input of the next.

**Pipeline:**
- **Stage 1:** Research Agent -- gathers information on a topic
- **Stage 2:** Analysis Agent -- processes and analyzes the research
- **Stage 3:** Writing Agent -- produces a final polished output

Each stage uses GPT-4o via the OpenAI SDK. The key thing to notice
is how we pass structured JSON between each stage.

## Install Dependencies

In [ ]:
!pip install openai

## Imports and Configuration

Initialize the OpenAI client. The SDK will automatically use
the `OPENAI_API_KEY` environment variable, but we're explicit here
so it's clear where the key comes from.

In [ ]:
import json
import os

from openai import OpenAI


# Initialize the OpenAI client. The SDK will automatically use
# the OPENAI_API_KEY environment variable, but we're explicit here
# so it's clear where the key comes from.
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# We'll use GPT-4o for this demo.
MODEL = "gpt-4o"

## Stage 1: Research Agent

The Research Agent's job is to gather key facts about a topic.

- **Input:** A topic string (e.g., "Hawaii real estate market trends")
- **Output:** A structured JSON object with research findings

This is the FIRST stage -- it takes a simple string input and produces
structured data that the next stage can consume.

In [ ]:
def stage_1_research(topic: str) -> dict:
    """
    The Research Agent's job is to gather key facts about a topic.

    Input:  A topic string (e.g., "Hawaii real estate market trends")
    Output: A structured JSON object with research findings

    This is the FIRST stage — it takes a simple string input and produces
    structured data that the next stage can consume.
    """

    print("=" * 60)
    print("STAGE 1: Research Agent")
    print(f"Topic: {topic}")
    print("=" * 60)

    # The prompt instructs the LLM to return structured JSON.
    # This is critical — we need parseable output, not free-form text.
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"""You are a research agent. Research the following topic and return
your findings as a JSON object.

Topic: {topic}

Return ONLY valid JSON in this exact format (no markdown, no code fences):
{{
    "topic": "the topic",
    "key_findings": ["finding 1", "finding 2", "finding 3", "finding 4", "finding 5"],
    "statistics": ["stat 1", "stat 2", "stat 3"],
    "sources_referenced": ["source 1", "source 2"],
    "summary": "A 2-3 sentence summary of the research"
}}"""
            }
        ]
    )

    # Parse the LLM's response as JSON
    raw_output = response.choices[0].message.content

    try:
        research_data = json.loads(raw_output)
    except json.JSONDecodeError:
        # If the LLM didn't return valid JSON, wrap the text in a basic structure.
        # In production, you'd retry or have more robust parsing.
        print("WARNING: LLM did not return valid JSON. Wrapping in basic structure.")
        research_data = {
            "topic": topic,
            "key_findings": [raw_output],
            "statistics": [],
            "sources_referenced": [],
            "summary": raw_output[:200]
        }

    print(f"\nResearch complete. Found {len(research_data.get('key_findings', []))} key findings.")
    print(f"Output preview: {json.dumps(research_data, indent=2)[:300]}...")
    print()

    # This dict IS the output of Stage 1.
    # It will become the INPUT of Stage 2.
    return research_data

## Stage 2: Analysis Agent

The Analysis Agent's job is to process research and extract insights.

- **Input:** The `research_data` dict FROM Stage 1
- **Output:** A structured JSON object with analysis and recommendations

Notice: this function takes the EXACT output of `stage_1_research()`.
That's the pipeline pattern -- output of one becomes input of the next.

In [ ]:
def stage_2_analysis(research_data: dict) -> dict:
    """
    The Analysis Agent's job is to process research and extract insights.

    Input:  The research_data dict FROM Stage 1
    Output: A structured JSON object with analysis and recommendations

    Notice: this function takes the EXACT output of stage_1_research().
    That's the pipeline pattern — output of one becomes input of the next.
    """

    print("=" * 60)
    print("STAGE 2: Analysis Agent")
    print(f"Analyzing research on: {research_data.get('topic', 'unknown')}")
    print("=" * 60)

    # We serialize the Stage 1 output to JSON and embed it in the prompt.
    # This is how data flows between agents — as serialized JSON in prompts.
    research_json = json.dumps(research_data, indent=2)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"""You are an analysis agent. You have received research data from
a research agent. Analyze this data and provide insights.

RESEARCH DATA:
{research_json}

Analyze this research and return ONLY valid JSON in this exact format
(no markdown, no code fences):
{{
    "topic": "the topic",
    "main_insight": "The single most important takeaway",
    "trends_identified": ["trend 1", "trend 2", "trend 3"],
    "opportunities": ["opportunity 1", "opportunity 2"],
    "risks": ["risk 1", "risk 2"],
    "confidence_level": "high/medium/low",
    "recommended_actions": ["action 1", "action 2", "action 3"]
}}"""
            }
        ]
    )

    raw_output = response.choices[0].message.content

    try:
        analysis_data = json.loads(raw_output)
    except json.JSONDecodeError:
        print("WARNING: LLM did not return valid JSON. Wrapping in basic structure.")
        analysis_data = {
            "topic": research_data.get("topic", "unknown"),
            "main_insight": raw_output[:200],
            "trends_identified": [],
            "opportunities": [],
            "risks": [],
            "confidence_level": "low",
            "recommended_actions": []
        }

    print(f"\nAnalysis complete. Confidence: {analysis_data.get('confidence_level', 'unknown')}")
    print(f"Main insight: {analysis_data.get('main_insight', 'N/A')[:100]}...")
    print()

    # This dict IS the output of Stage 2.
    # It will become the INPUT of Stage 3.
    return analysis_data

## Stage 3: Writing Agent

The Writing Agent's job is to produce a polished, human-readable output.

- **Input:** The `analysis_data` dict FROM Stage 2
- **Output:** A final written report (plain text string)

This is the LAST stage -- it takes structured data and produces
a natural-language deliverable.

In [ ]:
def stage_3_writing(analysis_data: dict) -> str:
    """
    The Writing Agent's job is to produce a polished, human-readable output.

    Input:  The analysis_data dict FROM Stage 2
    Output: A final written report (plain text string)

    This is the LAST stage — it takes structured data and produces
    a natural-language deliverable.
    """

    print("=" * 60)
    print("STAGE 3: Writing Agent")
    print(f"Writing report on: {analysis_data.get('topic', 'unknown')}")
    print("=" * 60)

    # Again, we serialize the previous stage's output and embed it in the prompt.
    analysis_json = json.dumps(analysis_data, indent=2)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"""You are a professional writing agent. You have received analysis
data from an analysis agent. Write a polished, concise executive briefing
based on this analysis.

ANALYSIS DATA:
{analysis_json}

Write a professional executive briefing that:
1. Opens with the main insight
2. Covers the key trends and statistics
3. Highlights opportunities and risks
4. Ends with clear recommended actions

Format it as a clean, readable document with headers. Keep it concise
but thorough — about 300-400 words. Write in plain text (no markdown)."""
            }
        ]
    )

    final_report = response.choices[0].message.content

    print(f"\nReport complete. Length: {len(final_report)} characters")
    print()

    # The final output is a string — the finished deliverable.
    return final_report

## Pipeline Orchestrator

This is where you can see the pipeline pattern clearly:
```
research_data = stage_1(topic)
analysis_data = stage_2(research_data)   <-- uses stage 1 output
final_report  = stage_3(analysis_data)   <-- uses stage 2 output
```

Each stage's output flows directly into the next stage's input.

In [ ]:
def run_pipeline(topic: str) -> str:
    """
    Orchestrates the full 3-stage pipeline.

    This is where you can see the pipeline pattern clearly:
        research_data = stage_1(topic)
        analysis_data = stage_2(research_data)   <-- uses stage 1 output
        final_report  = stage_3(analysis_data)   <-- uses stage 2 output

    Each stage's output flows directly into the next stage's input.
    """

    print("\n" + "#" * 60)
    print(f"# PIPELINE START: {topic}")
    print("#" * 60 + "\n")

    # Stage 1: Research
    # Input: topic string
    # Output: research_data dict
    research_data = stage_1_research(topic)

    # Stage 2: Analysis
    # Input: research_data dict (from Stage 1)
    # Output: analysis_data dict
    analysis_data = stage_2_analysis(research_data)

    # Stage 3: Writing
    # Input: analysis_data dict (from Stage 2)
    # Output: final_report string
    final_report = stage_3_writing(analysis_data)

    print("#" * 60)
    print("# PIPELINE COMPLETE")
    print("#" * 60)
    print("\n" + "=" * 60)
    print("FINAL REPORT:")
    print("=" * 60)
    print(final_report)

    return final_report

## Run the Pipeline

Run the pipeline with a sample topic. Change the `topic` variable to any topic you want to research.

In [ ]:
# Run the pipeline with a sample topic.
# Change this to any topic you want to research.
topic = "Current trends in the Hawaii residential real estate market"

report = run_pipeline(topic)

# Optionally save the report to a file
# with open("report_output.txt", "w") as f:
#     f.write(report)